In [1]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.decomposition import NMF

import gradio as gr

In [15]:
df = pd.read_csv(r"C:\Users\DELL\Downloads\Reviews.csv")

# Use only important columns
df = df[['Review Text', 'Rating', 'Recommended IND']]
df.dropna(inplace=True)

df.head(10)

,Review Text,Rating,Recommended IND
0,Absolutely wonderful - silky and sexy and comf...,4,1
1,Love this dress! it's sooo pretty. i happene...,5,1
2,I had such high hopes for this dress and reall...,3,0
3,"I love, love, love this jumpsuit. it's fun, fl...",5,1
4,This shirt is very flattering to all due to th...,5,1
5,"I love tracy reese dresses, but this one is no...",2,0
6,I aded this in my basket at hte last mintue to...,5,1
7,"I ordered this in carbon for store pick up, an...",4,1
8,I love this dress. i usually get an xs but it ...,5,1
9,"I'm 5""5' and 125 lbs. i ordered the s petite t...",5,1


In [4]:
def label_sentiment(rating):
    if rating >= 4:
        return "Positive"
    elif rating == 3:
        return "Neutral"
    else:
        return "Negative"

df['Sentiment'] = df['Rating'].apply(label_sentiment)

In [7]:
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    tokens = text.split()   # ← simple tokenization

    tokens = [lemmatizer.lemmatize(word) 
              for word in tokens 
              if word not in stop_words]

    return " ".join(tokens)

df['Clean_Text'] = df['Review Text'].apply(preprocess)

print("Before:", df['Review Text'].iloc[0])
print("After:", df['Clean_Text'].iloc[0])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Before: Absolutely wonderful - silky and sexy and comfortable
After: absolutely wonderful silky sexy comfortable


In [8]:
# Bag of Words
bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(df['Clean_Text'])

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(df['Clean_Text'])

y = df['Sentiment']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [10]:
model = LogisticRegression(max_iter=300)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

    Negative       0.56      0.42      0.48       457
     Neutral       0.50      0.23      0.31       588
    Positive       0.87      0.97      0.92      3484

    accuracy                           0.82      4529
   macro avg       0.64      0.54      0.57      4529
weighted avg       0.79      0.82      0.79      4529

Confusion Matrix:

[[ 192   75  190]
 [ 117  134  337]
 [  33   58 3393]]


In [11]:
def label_intent(text):
    text = text.lower()

    if "refund" in text or "return" in text:
        return "Refund Request"
    elif "late" in text or "delivery" in text or "shipping" in text:
        return "Delivery Issue"
    elif "bad" in text or "poor" in text or "damaged" in text:
        return "Complaint"
    else:
        return "General Query"

df['Intent'] = df['Review Text'].apply(label_intent)

In [12]:
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_tfidf, df['Intent'], test_size=0.2, random_state=42
)

intent_model = LogisticRegression(max_iter=300)
intent_model.fit(X_train_i, y_train_i)

intent_pred = intent_model.predict(X_test_i)

print(classification_report(y_test_i, intent_pred))

                precision    recall  f1-score   support

     Complaint       0.98      0.44      0.61        95
Delivery Issue       1.00      0.08      0.14        77
 General Query       0.96      1.00      0.98      4016
Refund Request       1.00      0.89      0.94       341

      accuracy                           0.96      4529
     macro avg       0.98      0.60      0.67      4529
  weighted avg       0.97      0.96      0.96      4529



In [13]:
nmf_model = NMF(n_components=3, random_state=42)
nmf_model.fit(X_tfidf)

feature_names = tfidf_vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(nmf_model.components_):
    print(f"\nTopic {topic_idx+1}:")
    print([feature_names[i] for i in topic.argsort()[-10:]])


Topic 1:
['run', 'large', 'ordered', 'fit', 'would', 'im', 'like', 'small', 'top', 'size']

Topic 2:
['im', 'slip', 'fabric', 'wear', 'fit', 'perfect', 'love', 'flattering', 'beautiful', 'dress']

Topic 3:
['wear', 'sweater', 'perfect', 'fit', 'soft', 'comfortable', 'jean', 'color', 'great', 'love']


In [14]:
def predict_review(review):
    clean = preprocess(review)
    vector = tfidf_vectorizer.transform([clean])

    sentiment = model.predict(vector)[0]
    intent = intent_model.predict(vector)[0]

    topic = nmf_model.transform(vector)
    topic_id = topic.argmax() + 1

    return sentiment, intent, f"Topic {topic_id}"

interface = gr.Interface(
    fn=predict_review,
    inputs="text",
    outputs=["text", "text", "text"],
    title="Customer Review Intelligence System"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv
